# Разведочный анализ PM2.5 (EDA)

**Цель.** Увидеть закономерности, которые потом проверим тестами (H1 — сезон, H2 — погода, H3 — суточный цикл), и оценить масштаб превышений нормы ВОЗ.

**Входы:** `data/processed/hourly.csv` (часы) и `data/processed/daily.csv` (сутки).
**Выходы:** графики в `figures/` (`00_pm25_distribution.png` и далее).

EDA — это разведка: мы смотрим и описываем, но не доказываем. Выводы подтвердим тестами в задаче 5.1, чтобы не «ловить звёздочки» на графиках.

**Зерно данных.** Суточные графики и статистика — по таблице `daily` (только сутки со средним `pm25_mean`); суточный цикл — по часовой таблице `hourly` (только часы со значением).

In [1]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import clean, eda, features

HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

WHO_LIMIT = features.WHO_DAILY_LIMIT
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]
HEATING_LABELS = {True: "Отопительный сезон", False: "Вне сезона"}

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Читаем таблицы и проверяем контракты

Читаем готовые таблицы функциями проекта и сразу проверяем их `check_*`: если контракт нарушен, дальше смотреть нечего. Суточные графики строим только по суткам, где есть среднее PM2.5, поэтому выделяем их в `days`.

In [2]:
hourly = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly)
daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

print("hourly:", hourly.shape)
print("daily:", daily.shape)

# Сутки со средним — с ними работают суточные графики
days = daily[daily["pm25_mean"].notna()]
print("суток со средним PM2.5:", len(days))

hourly: (49075, 18)
daily: (2044, 16)
суток со средним PM2.5: 1534


## Шаг 2. Сколько у нас данных

Средние и медианы pandas считает по непустым значениям. Мы не заполняем пропуски заново: короткие дыры уже заполнены на этапе очистки, длинные остаются честно пустыми. Поэтому важно знать, сколько значений стоит за каждым числом: сколько часов и суток пустые и как сутки со средним распределены по годам и месяцам.

In [3]:
hours_missing = hourly["pm25"].isna().sum()
print("часов без pm25:", hours_missing, f"({hours_missing / len(hourly):.2%})")
print("часов, заполненных интерполяцией:", hourly["pm25_interpolated"].sum())
print("суток со средним:", len(days), "| без среднего:", len(daily) - len(days))

# Сколько суток со средним в каждом году и месяце
days_by_year_month = days.assign(year=days["date"].dt.year).pivot_table(
    values="pm25_mean", index="year", columns="month", aggfunc="count"
)
print(days_by_year_month)

часов без pm25: 10235 (20.86%)
часов, заполненных интерполяцией: 1148
суток со средним: 1534 | без среднего: 510
month    1     2     3     4     5     6     7     8     9     10    11    12
year                                                                         
2020    NaN   NaN   NaN  17.0  20.0  17.0  18.0  22.0  24.0  17.0  23.0  23.0
2021   21.0  18.0  22.0  18.0  12.0   9.0   6.0  17.0  14.0   9.0  23.0  27.0
2022   20.0  26.0  26.0  26.0   NaN   NaN   NaN   NaN  22.0  28.0  28.0  20.0
2023   24.0   2.0   9.0  26.0  31.0  30.0  29.0  31.0  25.0  31.0  30.0  31.0
2024   26.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  26.0  22.0  30.0  31.0
2025   31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  28.0  31.0  13.0   NaN


**Что видим.** (вывод шага 2) Без `pm25` 10 235 часов из 49 075 (20.86 %), ещё 1 148 часов заполнены интерполяцией на этапе очистки. Суток со средним 1 534, без среднего 510. Главная дыра — май–август 2022: в таблице «год × месяц» там пусто, а в феврале–марте 2023 и ноябре 2025 мало суток (2, 9 и 13). Поэтому месячные статистики объединяют разное число лет, и это надо помнить при выводах.

## Шаг 3. Описательная статистика PM2.5

Считаем основные числа: среднее, медиану, квартили. PM2.5 скошен вправо (редкие пики смога), поэтому главные числа — медиана и межквартильный размах; среднее тянется за пиками. Сравниваем сезоны, отопительный сезон и месяцы.

In [4]:
print("Сутки, pm25_mean:")
print(days["pm25_mean"].describe())
print()
print("Часы, pm25:")
print(hourly["pm25"].describe())

Сутки, pm25_mean:
count    1534.000000
mean       29.065487
std        30.420802
min         3.289474
25%        11.010417
50%        16.871711
75%        32.526042
max       229.000000
Name: pm25_mean, dtype: float64

Часы, pm25:
count    38840.000000
mean        29.480947
std         37.746807
min          0.000000
25%          9.000000
50%         15.000000
75%         31.000000
max        421.000000
Name: pm25, dtype: float64


In [5]:
print("По сезонам (суточное среднее):")
print(days.groupby("season")["pm25_mean"].describe().reindex(SEASON_ORDER))
print()
print("По отопительному сезону:")
print(days.groupby("heating_season")["pm25_mean"].describe())

По сезонам (суточное среднее):
        count       mean        std        min        25%        50%  \
season                                                                 
winter  356.0  65.285704  40.476166  10.020833  32.757292  57.291667   
spring  391.0  17.597151  10.317504   4.021739  10.770833  14.375000   
summer  363.0  10.775799   3.751422   3.289474   8.458333  10.333333   
autumn  424.0  24.888314  18.397583   4.916667  13.875000  19.302083   

              75%         max  
season                         
winter  83.156250  229.000000  
spring  21.572917   79.416667  
summer  12.375000   33.625000  
autumn  29.489583  122.000000  

По отопительному сезону:
                count       mean        std       min        25%        50%  \
heating_season                                                                
False           763.0  12.617389   5.241042  3.289474   9.048611  11.375000   
True            771.0  45.342918  35.803281  4.500000  19.750000  32.208333   

 

In [6]:
print("По месяцам:")
print(days.groupby("month")["pm25_mean"].describe())

По месяцам:
       count       mean        std        min        25%        50%  \
month                                                                 
1      122.0  66.969343  43.870727  10.020833  32.354167  56.500000   
2      102.0  62.189044  29.424741  13.250000  46.223958  58.250000   
3      119.0  25.647255  13.094606   5.583333  16.687500  23.333333   
4      147.0  16.349315   6.595765   4.076389  11.458333  14.825000   
5      125.0  11.400908   4.203654   4.021739   8.583333  11.000000   
6      116.0  11.637772   4.418271   4.111111   8.645833  10.937500   
7      115.0  10.128128   3.414945   3.289474   7.979167   9.666667   
8      132.0  10.582567   3.246550   5.000000   8.657895  10.479167   
9      139.0  14.694215   5.239754   4.916667  10.718750  14.333333   
10     138.0  21.461171   8.767164   6.333333  15.062500  20.458333   
11     147.0  37.744951  24.619075   6.083333  20.354167  31.416667   
12     132.0  66.122487  44.538203  11.583333  29.934896  57.4583

In [7]:
# Скошенность: среднее больше медианы и положительный skew — длинный правый хвост
print("суточный pm25_mean: среднее", round(days["pm25_mean"].mean(), 1),
      "| медиана", round(days["pm25_mean"].median(), 1),
      "| skew", round(days["pm25_mean"].skew(), 2))
print("часовой pm25: среднее", round(hourly["pm25"].mean(), 1),
      "| медиана", round(hourly["pm25"].median(), 1),
      "| skew", round(hourly["pm25"].skew(), 2))

суточный pm25_mean: среднее 29.1 | медиана 16.9 | skew 2.48
часовой pm25: среднее 29.5 | медиана 15.0 | skew 2.91


**Что видим.** (вывод шага 3) Распределение скошено вправо: среднее суточного PM2.5 29.1 µg/m³ больше медианы 16.9, skew 2.48 (у часовых данных 29.5, 15.0 и 2.91). По сезонам медиана зимой 57.3 µg/m³, весной 14.4, летом 10.3, осенью 19.3. Отопительный сезон: медиана 32.2 µg/m³ (771 сутки), вне сезона 11.4 (763 суток); максимум суток 229 µg/m³. По месяцам самое высокое среднее в январе (67.0 µg/m³), самая высокая медиана в феврале (58.2), самые чистые месяцы — июль и август (медианы 9.7 и 10.5). Это описание; различие сезонов проверим тестом в 5.1.

## Шаг 4. Распределение: почему дальше Mann–Whitney

Слева — гистограмма суточного PM2.5 с линией нормы ВОЗ. Справа — ECDF (эмпирическая функция распределения): точка (x, y) значит «в доле y дней PM2.5 не превышал x»; кривая правее — значения выше. Две кривые — отопительный сезон и остальное время. ECDF сравнивает распределения целиком и не зависит от числа столбцов гистограммы. Если распределение скошено, сравнивать сезоны лучше непараметрическим тестом Mann–Whitney, а не t-тестом.

In [8]:
mean_pm25 = days["pm25_mean"].mean()
median_pm25 = days["pm25_mean"].median()
skew_pm25 = days["pm25_mean"].skew()
print(f"среднее {mean_pm25:.1f}, медиана {median_pm25:.1f}, skew {skew_pm25:.2f}")

fig, (ax_hist, ax_ecdf) = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(days["pm25_mean"], bins=50, ax=ax_hist)
ax_hist.axvline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax_hist.set_xlabel("PM2.5, µg/m³")
ax_hist.set_ylabel("Число дней")
ax_hist.legend()

sns.ecdfplot(data=days, x="pm25_mean", hue=days["heating_season"].map(HEATING_LABELS), ax=ax_ecdf)
ax_ecdf.axvline(WHO_LIMIT, color="black", linestyle="--")
ax_ecdf.set_xlabel("PM2.5, µg/m³")
ax_ecdf.set_ylabel("Доля дней с PM2.5 ≤ x")
ax_ecdf.get_legend().set_title("Период")

fig.suptitle(f"PM2.5 скошен вправо: среднее {mean_pm25:.0f} больше медианы {median_pm25:.0f} µg/m³")
fig.savefig(f"{FIGURES_DIR}/00_pm25_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)

среднее 29.1, медиана 16.9, skew 2.48


**Что видим.** (вывод шага 4) Гистограмма показывает длинный правый хвост: среднее 29.1 µg/m³ больше медианы 16.9, skew 2.48. Кривая ECDF отопительного сезона идёт правее кривой остального времени — в отопительный сезон значения выше. Раз распределение скошено, для сравнения сезонов в 5.1 берём непараметрический тест Mann–Whitney.

![График 0](../figures/00_pm25_distribution.png)

## Шаг 5. График 1 — как меняется загрязнение во времени

Вопрос: как PM2.5 меняется по датам. Для времени подходит линейный график: тонкая линия — сутки, толстая — среднее за 30 дней (сглаживает шум и показывает сезонность), пунктир — норма ВОЗ. Пропуски остаются разрывами линии: это честнее, чем рисовать выдуманные значения.

In [9]:
# Скользящее среднее: строка таблицы = календарные сутки подряд (проверено check_daily),
# поэтому окно 30 строк = 30 дней; если в окне меньше 15 дней со значением — среднее пустое
rolling_30 = daily["pm25_mean"].rolling(30, min_periods=15, center=True).mean()

peak_date = daily.loc[rolling_30.idxmax(), "date"]
peak_value = rolling_30.max()
print(f"максимум 30-дневного среднего: {peak_value:.1f} µg/m³, дата {peak_date.date()}")
print("максимум 30-дневного среднего по календарным годам:")
print(rolling_30.groupby(daily["date"].dt.year).max().round(1))

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily["date"], daily["pm25_mean"], linewidth=0.8, alpha=0.5, label="Сутки")
ax.plot(daily["date"], rolling_30, linewidth=2.5, label="Среднее за 30 дней")
ax.axhline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax.set_xlabel("Дата")
ax.set_ylabel("PM2.5, µg/m³")
ax.legend()
ax.set_title(f"Каждый год 30-дневное среднее PM2.5 выше нормы ВОЗ: до {peak_value:.0f} µg/m³ ({peak_value / WHO_LIMIT:.0f}× нормы)")
fig.savefig(f"{FIGURES_DIR}/01_timeseries.png", dpi=150, bbox_inches="tight")
plt.close(fig)


максимум 30-дневного среднего: 135.3 µg/m³, дата 2020-12-26
максимум 30-дневного среднего по календарным годам:
date
2020    135.3
2021    132.7
2022     83.7
2023    119.1
2024     61.5
2025     74.4
Name: pm25_mean, dtype: float64


**Что видим.** (вывод шага 5) Каждую зиму 30-дневное среднее поднимается далеко выше нормы ВОЗ: максимум 135.3 µg/m³ (около 9 норм) в конце декабря 2020. Максимумы по календарным годам: 2020 — 135.3, 2021 — 132.7, 2022 — 83.7, 2023 — 119.1, 2024 — 61.5, 2025 — 74.4 µg/m³, то есть во все годы выше 15. Летом среднее опускается к норме. Разрыв линии весной–летом 2022 года — это пропуск данных, а не чистый воздух.

![График 1](../figures/01_timeseries.png)

## Шаг 6. График 2 — какие месяцы хуже всего

Вопрос: как распределено суточное PM2.5 по месяцам. Для сравнения распределений по группам подходит boxplot: ящик — межквартильный размах, линия внутри — медиана. Выбросы не прячем: пики смога — это и есть явление, которое мы изучаем.

In [10]:
import warnings

# Предупреждение внутри seaborn 0.13 на matplotlib 3.11 (параметр vert), от нашего кода не зависит;
# глушим только его
warnings.filterwarnings("ignore", message="vert: bool")

month_medians = days.groupby("month")["pm25_mean"].median()
worst_month = month_medians.idxmax()
worst_median = month_medians.max()
print(month_medians.round(1))
print(f"худший месяц: {eda.MONTH_LABELS[worst_month]}, медиана {worst_median:.1f} µg/m³")

fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(data=days, x="month", y="pm25_mean", color=sns.color_palette()[0], ax=ax)
ax.axhline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax.set_xticks(range(12))
ax.set_xticklabels([eda.MONTH_LABELS[m] for m in range(1, 13)])
ax.set_xlabel("Месяц")
ax.set_ylabel("PM2.5 за сутки, µg/m³")
ax.legend()
ax.set_title(f"Хуже всего {eda.MONTH_LABELS[worst_month]}: медиана {worst_median:.0f} µg/m³ — в {worst_median / WHO_LIMIT:.1f} раза выше нормы ВОЗ")
fig.savefig(f"{FIGURES_DIR}/02_monthly_box.png", dpi=150, bbox_inches="tight")
plt.close(fig)


month
1     56.5
2     58.2
3     23.3
4     14.8
5     11.0
6     10.9
7      9.7
8     10.5
9     14.3
10    20.5
11    31.4
12    57.5
Name: pm25_mean, dtype: float64
худший месяц: фев, медиана 58.2 µg/m³


**Что видим.** (вывод шага 6) Медиана выше нормы ВОЗ в январе (56.5), феврале (58.2), марте (23.3), октябре (20.5), ноябре (31.4) и декабре (57.5 µg/m³); хуже всего февраль — 58.2 µg/m³, в 3.9 раза выше нормы. В апреле (14.8) и сентябре (14.3) медиана около нормы, с мая по август — ниже (9.7–11.0). Выбросы есть почти во всех месяцах: это отдельные дни смога. Месяцы объединяют разное число лет (дыра лета 2022).

![График 2](../figures/02_monthly_box.png)

## Шаг 7. График 7 — масштаб проблемы: доля дней выше нормы ВОЗ

Считаем долю суток со средним выше 15 µg/m³ по месяцам; месяцы сортируем по убыванию доли. Сутки без среднего не считаем: «нет данных» не значит «не превышено» (в таблице это `<NA>`, и `count`/`sum` его пропускают).

In [11]:
overall_share = days["who_exceed"].mean()
print(f"превышений: {days['who_exceed'].sum()} из {len(days)} суток ({overall_share:.1%})")

share = eda.who_exceed_share_by_month(daily)
print(share)
print("сумма days_known:", share["days_known"].sum())

# Три зимних месяца: доли из таблицы выше
winter_share = share[share["month"].isin([12, 1, 2])]["share"]
print(f"дек, янв, фев: доля от {winter_share.min():.1%} до {winter_share.max():.1%}")

worst = share.iloc[0]
best = share.iloc[-1]
print(f"худший месяц: {eda.MONTH_LABELS[worst['month']]} {worst['share']:.1%}, "
      f"лучший: {eda.MONTH_LABELS[best['month']]} {best['share']:.1%}")

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar([eda.MONTH_LABELS[m] for m in share["month"]], share["share"] * 100,
              color=sns.color_palette()[0])
ax.bar_label(bars, fmt="%.0f%%")
ax.set_xlabel("Месяц")
ax.set_ylabel("Доля дней с PM2.5 > 15 µg/m³, %")
ax.set_title(f"Норма ВОЗ превышена в {overall_share:.0%} дней; в дек, янв, фев — в {winter_share.mean():.0%}, в {eda.MONTH_LABELS[best['month']]} — в {best['share']:.0%}")
fig.savefig(f"{FIGURES_DIR}/07_who_exceed.png", dpi=150, bbox_inches="tight")
plt.close(fig)


превышений: 845 из 1534 суток (55.1%)
    month  days_known  days_exceed     share
0      12         132          127  0.962121
1       2         102           98  0.960784
2       1         122          117  0.959016
3      11         147          126  0.857143
4       3         119           94  0.789916
5      10         138          104  0.753623
6       4         147           72  0.489796
7       9         139           62  0.446043
8       6         116           16  0.137931
9       5         125           15      0.12
10      7         115            9  0.078261
11      8         132            5  0.037879
сумма days_known: 1534
дек, янв, фев: доля от 95.9% до 96.2%
худший месяц: дек 96.2%, лучший: авг 3.8%


**Что видим.** (вывод шага 7) Норма ВОЗ превышена в 845 сутках из 1 534 (55.1 %). Три зимних месяца почти одинаково плохи: декабрь 96.2 % дней (127 из 132), февраль 96.1 % (98 из 102), январь 95.9 % (117 из 122); лучше всего август — 4 % (5 из 132). Сумма суток со средним по месяцам равна 1 534. Сутки без среднего в долю не входят.

![График 7](../figures/07_who_exceed.png)

## Шаг 8. График 3 — когда в течение суток пик и зависит ли он от сезона

Вопрос: в какой час и в какой месяц PM2.5 выше всего. Для двух категорий (час × месяц) и одной величины подходит тепловая карта: цвет — средний PM2.5. Считаем по часовой таблице, час — местное время Алматы (признак `hour` из задачи 3.1, смена пояса 1 марта 2024 уже учтена). Таблицу строим через `pivot_table` (лекция 5). Рядом считаем, сколько часов стоит за каждой клеткой, чтобы знать, насколько ей можно верить.

In [12]:
table = eda.hour_month_table(hourly)
counts = eda.hour_month_table(hourly, aggfunc="count")
print("размер таблицы:", table.shape)
print("часов в клетке: минимум", counts.min().min(), ", максимум", counts.max().max())

peak_hour, peak_month = table.stack().idxmax()
peak_value = table.stack().max()
print(f"максимум: месяц {eda.MONTH_LABELS[peak_month]}, час {peak_hour}:00, {peak_value:.1f} µg/m³")
print("минимум таблицы:", round(table.stack().min(), 1), "µg/m³")

# Час максимума по каждому месяцу и профили зимы и июля — основа текста под графиком
print("час максимума по месяцам:")
print(table.idxmax())
print(f"июль: минимум {table[7].min():.1f}, максимум {table[7].max():.1f} µg/m³")
print("PM2.5 в 12–18 ч, декабрь и январь:")
print(table.loc[12:18, [12, 1]].round(0))
print("минимум по часам, декабрь и январь:")
print(table[[12, 1]].idxmin())

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(table.rename(columns=eda.MONTH_LABELS), cmap="viridis",
            cbar_kws={"label": "PM2.5, µg/m³"}, ax=ax)
ax.set_xlabel("Месяц")
ax.set_ylabel("Час (местное время Алматы)")
ax.set_title(f"Пик PM2.5 — после полудня в холодные месяцы: {eda.MONTH_LABELS[peak_month]}, {peak_hour}:00 — {peak_value:.0f} µg/m³")
fig.savefig(f"{FIGURES_DIR}/03_hour_month_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)


размер таблицы: (24, 12)
часов в клетке: минимум 107 , максимум 159
максимум: месяц янв, час 15:00, 95.3 µg/m³
минимум таблицы: 7.7 µg/m³
час максимума по месяцам:
month
1     15
2     13
3     11
4     10
5     20
6      9
7     20
8     20
9     19
10    18
11    15
12    14
dtype: int64
июль: минимум 8.0, максимум 12.8 µg/m³
PM2.5 в 12–18 ч, декабрь и январь:
month    12    1 
hour             
12     81.0  91.0
13     88.0  94.0
14     93.0  94.0
15     92.0  95.0
16     92.0  94.0
17     89.0  94.0
18     87.0  92.0
минимум по часам, декабрь и январь:
month
12    7
1     6
dtype: int64


**Что видим.** (вывод шага 8) Самое высокое среднее — январь, 15:00: 95.3 µg/m³; самое низкое в таблице — 7.7 µg/m³. Зимой (декабрь, январь) суточный цикл выражен: минимум рано утром (январь — 6 часов, декабрь — 7), с 12 до 18 часов значения 81–95 µg/m³ (по выводу под графиком), к ночи спад. В холодные месяцы час максимума после полудня: ноябрь — 15, декабрь — 14, январь — 15, февраль — 13. Летом (июль) цикл почти плоский: от 8.0 до 12.8 µg/m³ в любой час. Час максимума меняется по месяцам (от 9 часов в июне до 20 часов в мае, июле и августе), поэтому «один пик в сутки» для всех сезонов неверен. В каждой клетке от 107 до 159 часов, то есть карта опирается на достаточно большое число наблюдений. Это описание, проверка H3 — в задаче 5.1.

![График 3](../figures/03_hour_month_heatmap.png)

## Шаг 9. График 4 — суточный цикл в отопительный сезон и вне его

Вопрос: одинаков ли суточный цикл в отопительный сезон и вне его (H3). Для профиля по часам подходит линия по каждой группе. Вокруг линии рисуем 95 % доверительный интервал (лекция 8): `среднее ± t · std / √n`, то есть если повторять выборку много раз, 95 % таких интервалов накроют истинное среднее. Оговорка: соседние часы зависимы друг от друга, поэтому настоящий интервал шире нарисованного.

In [13]:
profile = eda.mean_with_ci(hourly, "pm25", ["heating_season", "hour"])
print("минимальное n в группе-часе:", profile["n"].min())

heating = profile[profile["heating_season"]].reset_index(drop=True)
other = profile[~profile["heating_season"]].reset_index(drop=True)
for name, group in [("отопительный сезон", heating), ("вне сезона", other)]:
    peak_row = group.loc[group["mean"].idxmax()]
    low_row = group.loc[group["mean"].idxmin()]
    print(f"{name}: пик {int(peak_row['hour'])}:00 — {peak_row['mean']:.1f}, минимум {int(low_row['hour'])}:00 — {low_row['mean']:.1f} µg/m³")
print("в сезон выше в каждый час:", (heating["mean"].values > other["mean"].values).all())
print("интервалы не пересекаются ни в один час:", (heating["ci_low"].values > other["ci_high"].values).all())

# Температура по часам в отопительный сезон: сверяем метки времени погоды
print("средняя температура по часам, отопительный сезон, °C:")
print(hourly[hourly["heating_season"]].groupby("hour")["temp_c"].mean().round(1))

peak_hour_heating = int(heating.loc[heating["mean"].idxmax(), "hour"])
peak_heating = heating["mean"].max()
peak_other = other["mean"].max()

fig, ax = plt.subplots(figsize=(14, 7))
for flag, group in [(True, heating), (False, other)]:
    ax.plot(group["hour"], group["mean"], marker="o", label=HEATING_LABELS[flag])
    ax.fill_between(group["hour"], group["ci_low"], group["ci_high"], alpha=0.25)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Час (местное время Алматы)")
ax.set_ylabel("Средний PM2.5, µg/m³")
ax.legend()
ax.set_title(f"В отопительный сезон пик PM2.5 в {peak_hour_heating}:00 — {peak_heating:.0f} µg/m³, вне сезона максимум {peak_other:.0f}")
fig.savefig(f"{FIGURES_DIR}/04_hourly_profile.png", dpi=150, bbox_inches="tight")
plt.close(fig)


минимальное n в группе-часе: 797
отопительный сезон: пик 14:00 — 61.1, минимум 6:00 — 27.7 µg/m³
вне сезона: пик 20:00 — 15.9, минимум 4:00 — 9.1 µg/m³
в сезон выше в каждый час: True
интервалы не пересекаются ни в один час: True
средняя температура по часам, отопительный сезон, °C:
hour
0     0.5
1     0.3
2     0.1
3     0.0
4    -0.2
5    -0.3
6    -0.4
7    -0.5
8    -0.0
9     1.1
10    2.4
11    3.5
12    4.4
13    5.0
14    5.4
15    5.4
16    4.9
17    4.0
18    3.1
19    2.4
20    2.0
21    1.5
22    1.1
23    0.8
Name: temp_c, dtype: float64


**Что видим.** (вывод шага 9) В отопительный сезон пик PM2.5 в 14:00 — 61.1 µg/m³, минимум в 6:00 — 27.7 µg/m³; вне сезона пик в 20:00 — 15.9, минимум в 4:00 — 9.1 µg/m³. В сезон среднее выше в каждый из 24 часов, и интервалы двух групп не пересекаются ни в один час. В каждой точке не меньше 797 часов. Оговорка: соседние часы зависимы, поэтому настоящие интервалы шире нарисованных; строгий ответ даст тест в задаче 5.1.

![График 4](../figures/04_hourly_profile.png)

## Шаг 10. График 5 — связь PM2.5 с ветром и температурой в отопительный сезон

Вопрос: как PM2.5 связан с ветром и температурой (H2). Берём только сутки отопительного сезона: за весь год температура почти повторяет сезон (холодно — значит отопление), и связь «холоднее — грязнее» оказалась бы просто связью «зима — грязнее» (это H1). Внутри сезона видно, добавляют ли что-то ветер и мороз. Подходит диаграмма рассеяния (две числовые величины). Линия тренда — медиана PM2.5 в 10 интервалах по оси X (`pd.qcut` ⚠️: интервалы с примерно равным числом дней), а не линия регрессии: регрессия на скошенных данных тянется за пиками. Связь меряем ρ Спирмена (лекция 9).

In [14]:
heating_days = days[days["heating_season"]]
print("суток отопительного сезона со средним:", len(heating_days))

wind_trend = eda.binned_medians(heating_days, "wind_speed", "pm25_mean")
temp_trend = eda.binned_medians(heating_days, "temp_c", "pm25_mean")
print("Ветер, медианы по интервалам:")
print(wind_trend)
print()
print("Температура, медианы по интервалам:")
print(temp_trend)

season_corr = eda.spearman_matrix(heating_days, ["pm25_mean", "wind_speed", "temp_c"])
rho_wind = season_corr.loc["pm25_mean", "wind_speed"]
rho_temp = season_corr.loc["pm25_mean", "temp_c"]
print()
print(f"ρ Спирмена с pm25_mean: ветер {rho_wind:.2f}, температура {rho_temp:.2f}")

fig, (ax_wind, ax_temp) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
panels = [
    (ax_wind, "wind_speed", wind_trend, f"Ветер: ρ = {rho_wind:.2f}", "Скорость ветра за сутки, м/с"),
    (ax_temp, "temp_c", temp_trend, f"Температура: ρ = {rho_temp:.2f}", "Температура за сутки, °C"),
]
for ax, column, trend, panel_title, x_label in panels:
    ax.scatter(heating_days[column], heating_days["pm25_mean"], alpha=0.4, s=15)
    ax.plot(trend["x_median"], trend["y_median"], marker="o", linewidth=2.5,
            color=sns.color_palette()[1], label="Медиана по интервалам")
    ax.axhline(WHO_LIMIT, color="black", linestyle="--", label=f"Норма ВОЗ {WHO_LIMIT} µg/m³")
    ax.set_title(panel_title)
    ax.set_xlabel(x_label)
ax_wind.set_ylabel("PM2.5 за сутки, µg/m³")
ax_wind.legend()
fig.suptitle(
    f"В отопительный сезон PM2.5 заметно связан с температурой (ρ = {rho_temp:.2f}), "
    f"с ветром связь слабая (ρ = {rho_wind:.2f})"
)
fig.savefig(f"{FIGURES_DIR}/05_weather_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)


суток отопительного сезона со средним: 771
Ветер, медианы по интервалам:
   x_median   y_median   n
0  0.649167  43.208333  78
1  0.970417  47.791667  77
2  1.122083  29.583333  77
3  1.282083  29.416667  77
4  1.401667  25.729167  77
5  1.510417  26.875000  77
6  1.644583  29.687500  77
7  1.808750  27.833333  77
8  2.056667  34.166667  78
9  2.449167  40.562500  76

Температура, медианы по интервалам:
    x_median   y_median   n
0  -9.450000  77.447917  78
1  -5.962500  72.083333  77
2  -3.083333  54.229167  77
3  -1.291667  46.979167  77
4   1.195833  26.875000  77
5   3.304167  33.375000  77
6   5.366667  28.000000  77
7   7.558333  24.000000  77
8  10.420833  21.583333  77
9  14.133333  20.791667  77

ρ Спирмена с pm25_mean: ветер -0.06, температура -0.55


**Что видим.** (вывод шага 10) В отопительный сезон (771 сутки со средним) PM2.5 тем выше, чем холоднее: ρ Спирмена с температурой −0.55, медиана падает с 77.4 µg/m³ в самом холодном интервале (около −9.5 °C) до 20.8 µg/m³ в самом тёплом (около 14.1 °C). С ветром связь слабая: ρ = −0.06, и она не монотонна — медиана 43.2 µg/m³ при самом слабом ветре (около 0.65 м/с), минимум 25.7 при около 1.40 м/с и снова 40.6 при самом сильном (около 2.45 м/с). Это связь, а не причина; влияние ветра и температуры на PM2.5 проверим в 5.1.

![График 5](../figures/05_weather_scatter.png)

## Шаг 11. График 6 — какие факторы сильнее всего связаны с PM2.5

Вопрос: какая погодная величина сильнее связана с PM2.5. Для матрицы связей подходит тепловая карта с числами. Берём ρ Спирмена, а не Пирсона: PM2.5 скошен вправо, связь может быть не линейной, а ранги устойчивы к пикам смога. Все пары считаются на одних и тех же сутках (строки с пропуском в любой колонке убираются). Корреляция не доказывает причину (лекция 9): это разведка, проверка будет в задаче 5.1.

In [15]:
FACTOR_COLUMNS = ["pm25_mean", "temp_c", "wind_speed", "humidity", "precip", "pressure"]
FACTOR_LABELS = {
    "pm25_mean": "PM2.5",
    "temp_c": "Температура",
    "wind_speed": "Ветер",
    "humidity": "Влажность",
    "precip": "Осадки",
    "pressure": "Давление",
}

corr = eda.spearman_matrix(daily, FACTOR_COLUMNS)
print("суток в расчёте:", daily[FACTOR_COLUMNS].dropna().shape[0])

# Столбец PM2.5 за весь год и только за отопительный сезон — рядом
pm25_year = corr["pm25_mean"].drop("pm25_mean")
pm25_season = eda.spearman_matrix(heating_days, FACTOR_COLUMNS)["pm25_mean"].drop("pm25_mean")
compare = pd.DataFrame({"весь год": pm25_year, "отопительный сезон": pm25_season})
print(compare.reindex(pm25_year.abs().sort_values(ascending=False).index).round(2))

# ρ с PM2.5 предыдущих суток: для 6.1 и про зависимость соседних дней
lag_corr = eda.spearman_matrix(daily, ["pm25_mean", "pm25_lag1"])
print("ρ pm25_mean с pm25_lag1:", round(lag_corr.loc["pm25_mean", "pm25_lag1"], 2))

strongest = pm25_year.abs().idxmax()
strongest_rho = pm25_year[strongest]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr.rename(index=FACTOR_LABELS, columns=FACTOR_LABELS),
    annot=True, fmt=".2f", vmin=-1, vmax=1, center=0, cmap="vlag",
    square=True, cbar_kws={"label": "ρ Спирмена"}, ax=ax,
)
ax.set_title(f"Сильнее всего с PM2.5 связана {FACTOR_LABELS[strongest].lower()}: ρ = {strongest_rho:.2f}")
fig.savefig(f"{FIGURES_DIR}/06_corr.png", dpi=150, bbox_inches="tight")
plt.close(fig)


суток в расчёте: 1534
            весь год  отопительный сезон
temp_c         -0.72               -0.55
pressure        0.59                0.28
humidity        0.34               -0.09
precip         -0.17               -0.47
wind_speed     -0.13               -0.06
ρ pm25_mean с pm25_lag1: 0.8


**Что видим.** (вывод шага 11) Расчёт на 1 534 сутках. За весь год сильнее всего с PM2.5 связана температура: ρ = −0.72, затем давление (0.59), влажность (0.34), осадки (−0.17) и ветер (−0.13). Только по отопительному сезону картина другая: температура −0.55, осадки −0.47, давление 0.28, влажность −0.09, ветер −0.06; то есть часть годовых связей — это сезон, а не погода сама по себе, а знак у влажности меняется. ρ с PM2.5 предыдущих суток равна 0.80: соседние сутки сильно зависимы, это важно для 6.1 и для независимости наблюдений в тестах. Корреляция не доказывает причину.

![График 6](../figures/06_corr.png)

## Итог

EDA по графикам 0–7 (числа — в выводе указанных шагов):
- График 0 (шаг 4): суточный PM2.5 скошен вправо (среднее 29.1 больше медианы 16.9 µg/m³, skew 2.48), кривая ECDF отопительного сезона правее.
- График 1 (шаг 5): каждую зиму 30-дневное среднее выше нормы ВОЗ, максимум 135.3 µg/m³ в конце декабря 2020; летом около нормы.
- График 2 (шаг 6): медиана выше нормы с октября по март, хуже всего февраль (58.2 µg/m³); с мая по август ниже нормы.
- График 3 (шаг 8): зимой суточный цикл выражен (максимум 95.3 µg/m³ в январе в 15:00), летом почти плоский.
- График 4 (шаг 9): в отопительный сезон среднее выше в каждый из 24 часов (пик 14:00 — 61.1, минимум 6:00 — 27.7 µg/m³), вне сезона пик 20:00 — 15.9.
- График 5 (шаг 10): в отопительный сезон связь с температурой заметная (ρ = −0.55), с ветром слабая и не монотонная (ρ = −0.06).
- График 6 (шаг 11): за весь год сильнее всего температура (ρ = −0.72); по отопительному сезону большинство связей слабее, а связь с осадками сильнее (−0.47 в сезон против −0.17 за год), у влажности меняется знак (0.34 → −0.09); ρ с предыдущими сутками 0.80.
- График 7 (шаг 7): норма ВОЗ превышена в 845 сутках из 1 534 (55.1 %), в декабре в 96 % дней, в августе в 4 %.

**Что проверим тестами в 5.1.**
- H1 (сезон) и H2 (ветер, температура): тест Welch и Mann–Whitney (распределение скошено, шаг 4), χ² «сезон × превышение нормы».
- H3: суточный цикл виден на графиках 3–4.

**Открытый вопрос.** Мы не знаем, почему профиль PM2.5 по часам имеет пик около 14:00 и минимум около 6:00. Метки времени погоды по местному времени проверены по выводу шага 9 (средняя температура в отопительный сезон: минимум в 6–7 часов, максимум в 14–15). Проверку меток времени станции см. в части 2 ниже.

**Ограничения.**
- Без `pm25` 20.86 % часов и 510 суток; дыра лета 2022 (май–август), поэтому месячные статистики объединяют разное число лет.
- Интервалы на графике 4 узкие, потому что соседние часы зависимы; настоящие шире.
- Корреляции за весь год смешаны с сезоном (график 6: большинство связей по отопительному сезону слабее, с осадками сильнее).
- Одна станция.

Раздел тестов (5.1) будет добавлен ниже.

# Часть 2. Проверка меток времени станции

**Зачем.** В части 1 мы увидели неожиданный суточный профиль в отопительный сезон: пик PM2.5 приходится на 14:00 и почти совпадает с пиком температуры (14–15 ч, шаг 9), тогда как у модели CAMS максимум вечером (20–21 ч). Прежде чем интерпретировать форму профиля или проверять гипотезу H3, нужно убедиться, что метки времени станции действительно записаны в UTC.

**Что проверяем:**
1. Согласуются ли метки станции с настоящим UTC (переход Алматы на UTC+5 1 марта 2024 против температуры и CAMS; лаг корреляции со CAMS по сезонам; фаза суточного профиля относительно температуры и CAMS).
2. Насколько устойчив суточный профиль по отдельным отопительным сезонам.
3. Что можно утверждать про гипотезу H3 в разделе 5.1.

**Как проверяем.** Сравниваем данные станции с двумя рядами, у которых время точно в UTC: температурой Open-Meteo и моделью CAMS (оба источника запрошены с `timezone=UTC`).

**Предсказания до расчётов:**
- **Метки = настоящий UTC:** максимум корреляции со CAMS около лага 0 (допуск ±1–2 ч: CAMS — модель со сглаживанием и возможным запаздыванием, а обозначение начала или конца часа даёт ±1 ч).
- **Метки = местное время, записанное как UTC:** значение с меткой `T` измерено в `T − 6` ч (с 2024-03-01 — в `T − 5` ч) → максимум корреляции сместится примерно на **+6 / +5 ч**; тогда настоящий местный час равен `hour_utc` (метка как есть), а не пересчитанному `hour`.
- **Метки сдвинуты на −6 ч (двойной перевод):** максимум корреляции окажется около **−6 ч**.
- Различить сдвиг ±1 ч (начало/конец часа) такими проверками нельзя.

**Источник.** По документации OpenAQ «Dates and datetimes» (`docs.openaq.org/using-the-api/dates-datetimes`), у измерения хранится пара меток — UTC и местная, а у старых источников метка может обозначать конец часа (для AirNow прямых указаний в документации нет).

**Важно.** В этой части таблицы `data/` не меняются — мы только проверяем и описываем данные. Если обнаружится сдвиг меток, исправление будет отдельной задачей.

## Шаг 12. Время каждого часа: местный час, час UTC, сезон

Переводим `time_utc` в местное время Алматы, выделяем час UTC (`hour_utc`), метку отопительного сезона (`2020/21` и т. д.) и квартал. Проверяем, что часовая сетка полная (шаг везде 1 ч — это нужно для корректных сдвигов по времени), фиксируем час перехода Алматы на UTC+5 (`switch_utc`) и отбираем отопительные сезоны, в которых в каждом местном часе есть не меньше `MIN_VALUES_PER_HOUR = 60` измерений `pm25` (примерно два месяца данных).

In [16]:
MIN_VALUES_PER_HOUR = 60

local_time = clean.to_almaty_time(hourly["time_utc"])
checks = hourly.assign(
    hour_utc=hourly["time_utc"].dt.hour,
    season_label=eda.heating_season_label(local_time),
    quarter=local_time.dt.year.astype(str) + "Q" + local_time.dt.quarter.astype(str),
)

# Проверяем полноту часовой сетки: шаг между соседними строками должен быть ровно 1 ч
steps = checks["time_utc"].diff().dropna()
print("сетка часов полная (шаг везде 1 ч):", (steps == pd.Timedelta(hours=1)).all())

# Смещение «местное время − UTC» и первый час после перехода на UTC+5
offset = local_time.dt.tz_localize(None) - hourly["time_utc"].dt.tz_localize(None)
print("число часов по смещению (местное − UTC):")
print(offset.value_counts().sort_index())

switch_utc = checks.loc[offset == pd.Timedelta(hours=5), "time_utc"].min()
print("switch_utc (первый час с UTC+5):", switch_utc)

# Отбираем отопительные сезоны с достаточным числом наблюдений в каждом местном часе
heating = checks[checks["heating_season"]]
counts = eda.hour_profile_table(heating, "pm25", "season_label", aggfunc="count")
print("минимум значений pm25 в часе по сезонам:")
print(counts.min())

SEASONS = list(counts.columns[counts.min() >= MIN_VALUES_PER_HOUR])
print(f"отобранные сезоны (минимум >= {MIN_VALUES_PER_HOUR}):", SEASONS)

сетка часов полная (шаг везде 1 ч): True
число часов по смещению (местное − UTC):
time_utc
0 days 05:00:00    14969
0 days 06:00:00    34106
Name: count, dtype: int64
switch_utc (первый час с UTC+5): 2024-02-29 18:00:00+00:00
минимум значений pm25 в часе по сезонам:
season_label
2019/20      6
2020/21    138
2021/22    142
2022/23    121
2023/24    177
2024/25    182
2025/26     30
dtype: int64
отобранные сезоны (минимум >= 60): ['2020/21', '2021/22', '2022/23', '2023/24', '2024/25']


**Что видим.** (вывод шага 12) Часовая сетка полная (шаг везде 1 ч). Смещение «местное − UTC» равно +6 ч для 34 106 часов и +5 ч для 14 969 часов; первый час со смещением +5 ч (`switch_utc`) — `2024-02-29 18:00:00+00:00`. Порог `MIN_VALUES_PER_HOUR = 60` проходят пять отопительных сезонов: `2020/21` (минимум 138 значений в часе), `2021/22` (142), `2022/23` (121), `2023/24` (177) и `2024/25` (182). Сезоны `2019/20` (минимум 6 значений) и `2025/26` (минимум 30 значений) отброшены как неполные.

## Шаг 13. График 11 — один ли суточный профиль во все отопительные сезоны

Проверяем, повторяется ли суточный профиль (пик в 14:00 и минимум в 6:00 из шага 9) в каждом отопительном сезоне отдельно. Зимой дневные значения образуют широкое плато, и сырой `idxmax` может прыгать из-за одного случайного часа. Поэтому мы считаем **сглаженный по кругу профиль** ⚠️ (`smooth_profile` с окном 3 ч через `np.roll`: после 23 ч идёт 0 ч, берётся среднее часа и двух соседей) и находим его экстремумы (`profile_extremes`), а рядом печатаем сырой `idxmax`. Сходство формы профилей разных сезонов оцениваем корреляцией Пирсона по 24 точкам. Чтобы сравнить форму независимо от общего уровня смога в разные годы, на правой панели делим профиль каждого сезона на его среднее.

In [17]:
profiles = eda.hour_profile_table(heating, "pm25", "season_label")[SEASONS]
extremes = eda.profile_extremes(profiles)
extremes["peak_to_low"] = extremes["peak"] / extremes["low"]
print("экстремумы сглаженного профиля (окно 3 ч):")
print(extremes.round(1))
print("сырой час максимума (idxmax):")
print(profiles.idxmax())

shape_corr = profiles.corr()
print("корреляция Пирсона между профилями сезонов:")
print(shape_corr.round(2))

# Берём только пары разных сезонов (без диагонали, где всегда 1.0)
pair_corrs = []
for i, season_a in enumerate(SEASONS):
    for season_b in SEASONS[i + 1 :]:
        pair_corrs.append((shape_corr.loc[season_a, season_b], season_a, season_b))
min_corr, min_a, min_b = min(pair_corrs)
max_corr, max_a, max_b = max(pair_corrs)
print(
    f"сходство формы вне диагонали: min {min_corr:.3f} ({min_a} и {min_b}), "
    f"max {max_corr:.3f} ({max_a} и {max_b})"
)

# Нормированный профиль: отношение к среднему за сезон показывает форму независимо от уровня
relative = profiles / profiles.mean()

first_season = SEASONS[0]
last_season = SEASONS[-1]
first_peak = int(extremes.loc[first_season, "peak_hour"])
last_peak = int(extremes.loc[last_season, "peak_hour"])
low_min = int(extremes["low_hour"].min())
low_max = int(extremes["low_hour"].max())

fig, (ax_raw, ax_rel) = plt.subplots(1, 2, figsize=(16, 6))
for season_name in SEASONS:
    ax_raw.plot(profiles.index, profiles[season_name], marker="o", label=season_name)
    ax_rel.plot(relative.index, relative[season_name], marker="o", label=season_name)

for ax in (ax_raw, ax_rel):
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Час (местное время Алматы)")
    ax.legend(title="Сезон")

ax_raw.set_ylabel("Средний PM2.5, µg/m³")
ax_rel.set_ylabel("PM2.5 / среднее за сезон")

fig.suptitle(
    f"Час пика PM2.5 в отопительный сезон смещается с {first_peak}:00 ({first_season}) "
    f"до {last_peak}:00 ({last_season}), минимум остаётся в {low_min}–{low_max} ч (сглаженный профиль)"
)
fig.savefig(f"{FIGURES_DIR}/11_profile_by_year.png", dpi=150, bbox_inches="tight")
plt.close(fig)

экстремумы сглаженного профиля (окно 3 ч):
              peak_hour  peak  low_hour   low  peak_to_low
season_label                                              
2020/21              20  80.7         7  47.3          1.7
2021/22              17  61.6         6  26.1          2.4
2022/23              15  71.5         6  29.9          2.4
2023/24              16  52.4         5  19.9          2.6
2024/25              14  60.2         5  22.7          2.7
сырой час максимума (idxmax):
season_label
2020/21    19
2021/22    17
2022/23    16
2023/24    16
2024/25    13
dtype: int64
корреляция Пирсона между профилями сезонов:
season_label  2020/21  2021/22  2022/23  2023/24  2024/25
season_label                                             
2020/21          1.00     0.83     0.87     0.77     0.51
2021/22          0.83     1.00     0.98     0.99     0.87
2022/23          0.87     0.98     1.00     0.98     0.82
2023/24          0.77     0.99     0.98     1.00     0.90
2024/25          0.51     

**Что видим.** (вывод шага 13) Суточный профиль меняется от сезона к сезону. По сглаженному профилю (окно 3 ч) час пика смещается с вечера на ранний день: `2020/21` — 20:00 (80.7 µg/m³, сырой `idxmax` 19), `2021/22` — 17:00 (61.6, сырой 17), `2022/23` — 15:00 (71.5, сырой 16), `2023/24` — 16:00 (52.4, сырой 16), `2024/25` — 14:00 (60.2, сырой 13). Минимум держится рано утром в 5–7 ч (`2020/21` — 7:00 и 47.3 µg/m³, `2021/22` — 6:00 и 26.1, `2022/23` — 6:00 и 29.9, `2023/24` — 5:00 и 19.9, `2024/25` — 5:00 и 22.7). Отношение `peak_to_low` выросло с 1.7 в `2020/21` до 2.4–2.7 в последующих сезонах. По корреляции Пирсона сезоны `2021/22`–`2023/24` почти одинаковы по форме (0.98–0.99, максимум 0.988 между `2021/22` и `2023/24`), а минимальное сходство — 0.515 между `2020/21` и `2024/25`.

![График 11](../figures/11_profile_by_year.png)

## Шаг 14. Когда менялся профиль: пик и минимум по часам UTC по кварталам

Чтобы понять, в какой момент изменилась форма суточного профиля, смотрим на экстремумы по **часам UTC** (`hour_utc`) в каждом квартале. Час UTC — это метка времени как есть: он не зависит от перехода Казахстана на UTC+5 в 2024 году и от предположений о часовом поясе. Берём все часы (а не только отопительный сезон), чтобы проследить все кварталы подряд, и отсекаем неполные кварталы по порогу `MIN_VALUES_PER_QUARTER = 800` измерений `pm25`.

In [18]:
MIN_VALUES_PER_QUARTER = 800

quarter_counts = checks.groupby("quarter")["pm25"].count()
QUARTERS = list(quarter_counts[quarter_counts >= MIN_VALUES_PER_QUARTER].index)
dropped_quarters = quarter_counts[quarter_counts < MIN_VALUES_PER_QUARTER]

utc_profiles = eda.hour_profile_table(
    checks, "pm25", "quarter", hour_column="hour_utc"
)[QUARTERS]
quarter_extremes = eda.profile_extremes(utc_profiles)
quarter_extremes["n"] = quarter_counts[QUARTERS]
quarter_extremes["peak_to_low"] = quarter_extremes["peak"] / quarter_extremes["low"]

print(f"экстремумы по часам UTC (кварталы с n >= {MIN_VALUES_PER_QUARTER}):")
with pd.option_context("display.max_rows", None, "display.width", 120):
    print(quarter_extremes.round(1))

print(f"\nотброшенные кварталы (n < {MIN_VALUES_PER_QUARTER}):")
print(dropped_quarters)

экстремумы по часам UTC (кварталы с n >= 800):
         peak_hour  peak  low_hour   low     n  peak_to_low
quarter                                                    
2020Q2          15  15.4        23   6.7  1513          2.3
2020Q3          14  15.5        23   8.1  1780          1.9
2020Q4          14  80.9         0  48.3  1721          1.7
2021Q1          15  81.4         1  45.3  1766          1.8
2021Q2           4  19.9        23   9.7  1181          2.0
2021Q3          14  15.3        22   8.8  1101          1.7
2021Q4          12  74.2         0  29.0  1507          2.6
2022Q1           9  57.5         0  24.3  1823          2.4
2022Q4           9  63.5         0  25.9  1922          2.4
2023Q1          11  94.4         1  38.0   968          2.5
2023Q2           5  17.0        23  10.7  2122          1.6
2023Q3          14  13.2        22   7.6  2073          1.7
2023Q4          10  45.7        23  16.5  2208          2.8
2024Q1           8  62.1         0  23.4  2036       

**Что видим.** (вывод шага 14) Порог `MIN_VALUES_PER_QUARTER = 800` прошли 21 квартал (`n` от 968 в `2023Q1` до 2208 в `2023Q4`); отброшены только два квартала 2022 года — `2022Q2` (`n = 668`) и `2022Q3` (`n = 552`).

- **Осенне-зимние кварталы (`Q4`, `Q1`):** час пика по UTC смещается с **14–15 UTC** в `2020Q4`–`2021Q1` (14 в `2020Q4`, 15 в `2021Q1`) до **12 UTC** в `2021Q4`, а с `2022Q1` по `2025Q4` держится в диапазоне **8–11 UTC** (9 в `2022Q1` и `2022Q4`, 11 в `2023Q1`, 10 в `2023Q4`, 8 в `2024Q1`, 9 в `2024Q4`, 8 в `2025Q1` и `2025Q4`).
- **Минимум по UTC стабилен** во все кварталы: **21–1 UTC**, а в осенне-зимних кварталах — **23–1 UTC** (чаще всего 0 UTC).
- **Связь с пропуском в `2022Q2`–`2022Q3` (факт по таблице):** сдвиг пика до 12 UTC (`2021Q4`) и до 9 UTC (`2022Q1`) произошёл ещё до отброшенных кварталов `2022Q2`–`2022Q3`, а после них пик остался на 8–11 UTC. То есть изменение началось до этого пропуска, а не возникло скачком на нём.
- **Тёплые кварталы (`Q2`, `Q3`):** перепад `peak_to_low` мал — от 1.6 (`2023Q2`, `2025Q2`, `2025Q3`) до 2.3 (`2020Q2`), значения пика 13.2–19.9 µg/m³, а час пика скачет от 4 (`2021Q2`) и 5 (`2023Q2`) до 13–15 UTC — на почти плоском летнем профиле час пика почти ничего не значит.

## Шаг 15. Переход Алматы на UTC+5 1 марта 2024

1 марта 2024 года Алматы перешёл с UTC+6 на UTC+5 (часы перевели на 1 час назад). Если метки станции записаны в настоящем UTC, то:
- у контрольных рядов с точным UTC (температура Open-Meteo и модель CAMS, суточный ход которых задаётся солнцем) пик по местному времени должен наступить примерно на **1 час раньше** (ожидаемый сдвиг `−1` ч);
- у станции `pm25` сдвиг ожидается около `0` ч (если суточный профиль определяется распорядком города по местным часам) или `−1` ч (если профиль задаётся солнцем и погодой); оба значения совместимы с настоящим UTC.

Берём узкое окно по 29 местных суток до и после `switch_utc` (`2024-02-01` … последний час до перехода и `switch_utc` … `2024-03-29`), чтобы не смешивать разные годы и многолетний дрейф профиля из шага 14. Температура служит контролем, так как между февралём и мартом меняется и долгота дня. Для сравнения приводим также сдвиг за весь отопительный период до и после `switch_utc`.

In [19]:
print("переход на UTC+5 (switch_utc):", switch_utc)

dates = features.local_dates(checks["time_utc"])
before = checks[(dates >= "2024-02-01") & (checks["time_utc"] < switch_utc)]
after = checks[(checks["time_utc"] >= switch_utc) & (dates <= "2024-03-29")]

print(
    "окно 29 суток до перехода — часов:",
    len(before),
    "| мин. pm25 в часе:",
    before.groupby("hour")["pm25"].count().min(),
)
print(
    "окно 29 суток после перехода — часов:",
    len(after),
    "| мин. pm25 в часе:",
    after.groupby("hour")["pm25"].count().min(),
)


def summarize_shifts(df_before: pd.DataFrame, df_after: pd.DataFrame) -> pd.DataFrame:
    """Лучший круговой сдвиг профиля (до -> после) для температуры, CAMS и PM2.5."""
    rows = []
    for column in ["temp_c", "cams_pm25", "pm25"]:
        profile_before = df_before.groupby("hour")[column].mean()
        profile_after = df_after.groupby("hour")[column].mean()
        shifts = eda.profile_shift(profile_before, profile_after, max_shift=6)
        best = shifts.loc[shifts["corr"].idxmax()]
        corr_zero = shifts.loc[shifts["shift"] == 0, "corr"].iloc[0]
        rows.append(
            {
                "column": column,
                "best_shift": int(best["shift"]),
                "best_corr": round(float(best["corr"]), 3),
                "corr_shift_0": round(float(corr_zero), 3),
            }
        )
    return pd.DataFrame(rows).set_index("column")


switch_table = summarize_shifts(before, after)
print("\nсдвиг профиля в узком окне (29 суток до и после):")
print(switch_table)

pm25_shifts = eda.profile_shift(
    before.groupby("hour")["pm25"].mean(),
    after.groupby("hour")["pm25"].mean(),
    max_shift=6,
)
print("\nкорреляция профилей pm25 по сдвигам в узком окне:")
print(pm25_shifts.round(3).to_string(index=False))

# Сравниваем со всем отопительным периодом до и после перехода, чтобы увидеть эффект дрейфа
heating_before = heating[heating["time_utc"] < switch_utc]
heating_after = heating[heating["time_utc"] >= switch_utc]
whole_table = summarize_shifts(heating_before, heating_after)
print("\nсдвиг профиля за весь отопительный период (до и после switch_utc):")
print(whole_table)

переход на UTC+5 (switch_utc): 2024-02-29 18:00:00+00:00
окно 29 суток до перехода — часов: 696 | мин. pm25 в часе: 28
окно 29 суток после перехода — часов: 697 | мин. pm25 в часе: 28

сдвиг профиля в узком окне (29 суток до и после):
           best_shift  best_corr  corr_shift_0
column                                        
temp_c             -1      0.973         0.877
cams_pm25          -1      0.988         0.908
pm25               -2      0.942         0.724

корреляция профилей pm25 по сдвигам в узком окне:
 shift   corr
    -6  0.288
    -5  0.530
    -4  0.749
    -3  0.893
    -2  0.942
    -1  0.871
     0  0.724
     1  0.534
     2  0.348
     3  0.178
     4  0.009
     5 -0.153
     6 -0.322

сдвиг профиля за весь отопительный период (до и после switch_utc):
           best_shift  best_corr  corr_shift_0
column                                        
temp_c             -1      0.995         0.945
cams_pm25          -1      0.912         0.856
pm25               -2      

**Что видим.** (вывод шага 15) В узком 29-суточном окне вокруг `2024-02-29 18:00:00+00:00` до перехода 696 часов, после перехода 697 часов; минимальное число наблюдений `pm25` на один местный час — 28 в обоих окнах.

- **Контрольные ряды** ведут себя ровно по предсказанию: у `temp_c` лучший сдвиг **−1 ч** (`corr = 0.973` против `0.877` при 0), у `cams_pm25` — **−1 ч** (`0.988` против `0.908`).
- **Станция `pm25`** в узком окне даёт лучший сдвиг **−2 ч** (`corr = 0.942`, при `−1` ч — `0.871`, при `0` ч — `0.724`). Сдвиг **−2 ч** у станции не совпадает ни с одним предсказанием для верного UTC (`0` или `−1` ч) и остаётся необъяснённым расхождением; результат нужно сопоставлять с шагами 16 и 17.
- **Весь отопительный период до и после `switch_utc`** даёт похожую картину (`temp_c` −1 ч и `0.995`, `cams_pm25` −1 ч и `0.912`, `pm25` −2 ч и `0.988` против `0.792` при 0). Однако опираться на весь период нельзя: до перехода объединены несколько отопительных сезонов, в которых профиль уже дрейфовал (шаг 14), поэтому многолетнее смещение пика накладывается на перевод часов.

## Шаг 16. График 12 — совпадают ли по времени станция и CAMS

Проверяем метки времени станции с помощью **кросс-корреляции по лагам** ⚠️: сдвигаем ряд CAMS (`cams_pm25`, у которого время точно в UTC) на `lag` часов с помощью `shift` и на каждом сдвиге считаем ρ Спирмена между `pm25` и сдвинутым рядом CAMS (`eda.lagged_correlations`). Используем именно Спирмена, так как часовые концентрации PM2.5 сильно скошены вправо и редкие пики смога искажали бы корреляцию Пирсона. Сдвиг выполняется на полной часовой сетке до отбора строк по периодам, чтобы на стыках сезонов не соединялись далёкие по времени часы.

Знак лага: `lag = +k` означает, что станция повторяет CAMS с опозданием на `k` часов (`cams.shift(lag)` в момент `t` берёт значение из `t − k`); `lag = −k` — станция опережает CAMS на `k` часов.

**Предсказания до расчётов:**
- Если метки станции записаны в настоящем UTC → максимум ρ около `0` (±1–2 ч).
- Если метки — местное время, записанное как UTC → максимум около `+6` ч (или `+5` ч после марта 2024).
- Если метки сдвинуты на `−6` ч (двойной перевод) → максимум около `−6` ч.

Считаем два варианта рядов: (1) исходные часовые ряды и (2) ряды, сглаженные скользящим средним за 24 часа (`rolling(24, center=True, min_periods=18).mean()` на полной сетке до фильтрации). Второй вариант убирает суточные циклы станции и CAMS и позволяет проверить, совпадает ли время многодневных эпизодов смога. На график 12 выводим только часовые ряды (одна панель), а числа по сглаженному 24-часовому варианту печатаем отдельно.

In [20]:
LAGS = range(-12, 13)

# Сглаживаем на полной часовой сетке до отбора строк по периодам
checks["pm25_24h"] = checks["pm25"].rolling(24, center=True, min_periods=18).mean()
checks["cams_24h"] = checks["cams_pm25"].rolling(24, center=True, min_periods=18).mean()

periods = {"весь период CAMS": checks["cams_pm25"].notna()}
cams_seasons = []
for label in SEASONS:
    mask = checks["heating_season"] & (checks["season_label"] == label)
    if checks.loc[mask, "cams_pm25"].notna().all():
        periods[f"сезон {label}"] = mask
        cams_seasons.append(label)
print("сезоны из SEASONS, где есть CAMS:", cams_seasons)

variants = [
    ("часовые", "pm25", "cams_pm25"),
    ("сглаженные 24 ч", "pm25_24h", "cams_24h"),
]

hourly_curves = {}
smoothed_ranges = {}
summary_rows = []
for period_name, mask in periods.items():
    for variant_name, x_col, y_col in variants:
        table = eda.lagged_correlations(checks, x_col, y_col, LAGS, rows=mask)
        if variant_name == "часовые":
            hourly_curves[period_name] = table
        else:
            smoothed_ranges[period_name] = (table["rho"].min(), table["rho"].max())
        best = table.loc[table["rho"].idxmax()]
        by_lag = table.set_index("lag")
        summary_rows.append(
            {
                "period": period_name,
                "variant": variant_name,
                "n": int(by_lag.loc[0, "n"]),
                "best_lag": int(best["lag"]),
                "best_rho": float(best["rho"]),
                "rho_minus_6": float(by_lag.loc[-6, "rho"]),
                "rho_0": float(by_lag.loc[0, "rho"]),
                "rho_plus_6": float(by_lag.loc[6, "rho"]),
            }
        )

lag_summary = pd.DataFrame(summary_rows)
print("\nсводка корреляции по лагам (n указано на лаге 0):")
print(lag_summary.round(3).to_string(index=False))

all_24h_min, all_24h_max = smoothed_ranges["весь период CAMS"]
print(
    f"\nвариант 24 ч за весь период CAMS: ρ по всем лагам −12…+12 лежит в узком диапазоне "
    f"{all_24h_min:.3f}–{all_24h_max:.3f} (почти плоский, версии меток не различает)"
)

all_hourly = lag_summary[
    (lag_summary["period"] == "весь период CAMS") & (lag_summary["variant"] == "часовые")
].iloc[0]

fig, ax = plt.subplots(figsize=(14, 6))
for period_name, curve in hourly_curves.items():
    ax.plot(curve["lag"], curve["rho"], marker="o", label=period_name)

ax.axvline(0, color="black", linestyle="-", label="Лаг 0 (метки в UTC)")
ax.axvline(
    -6,
    color="gray",
    linestyle="--",
    label="±6 ч (так было бы при сдвиге меток на 6 ч)",
)
ax.axvline(6, color="gray", linestyle="--")
ax.set_xticks(range(-12, 13, 3))
ax.set_xlabel("Сдвиг CAMS относительно станции, ч")
ax.set_ylabel("ρ Спирмена")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_title(
    f"Связь со CAMS при −6 ч (ρ = {all_hourly['rho_minus_6']:.3f}) близка к 0 ч "
    f"({all_hourly['rho_0']:.3f}), но падает при +6 ч ({all_hourly['rho_plus_6']:.3f}): "
    "версию «−6» не отличить"
)
fig.savefig(f"{FIGURES_DIR}/12_lag_check.png", dpi=150, bbox_inches="tight")
plt.close(fig)


сезоны из SEASONS, где есть CAMS: ['2022/23', '2023/24', '2024/25']



сводка корреляции по лагам (n указано на лаге 0):
          period         variant     n  best_lag  best_rho  rho_minus_6  rho_0  rho_plus_6
весь период CAMS         часовые 25780        -2     0.556        0.531  0.534       0.377
весь период CAMS сглаженные 24 ч 25561        -2     0.772        0.768  0.770       0.754
   сезон 2022/23         часовые  2932        -1     0.452        0.363  0.449       0.284
   сезон 2022/23 сглаженные 24 ч  2822        -4     0.598        0.598  0.595       0.578
   сезон 2023/24         часовые  4268        -1     0.466        0.387  0.460       0.172
   сезон 2023/24 сглаженные 24 ч  4251         1     0.638        0.613  0.638       0.623
   сезон 2024/25         часовые  4382        -3     0.477        0.451  0.451       0.190
   сезон 2024/25 сглаженные 24 ч  4392        -3     0.696        0.693  0.693       0.672

вариант 24 ч за весь период CAMS: ρ по всем лагам −12…+12 лежит в узком диапазоне 0.728–0.772 (почти плоский, версии меток не раз

**Что видим.** (вывод шага 16) В анализ со CAMS вошли три отопительных сезона: `2022/23`, `2023/24` и `2024/25` (в `2020/21` и `2021/22` ряд CAMS ещё пуст).

- **Часовые ряды:** за весь период CAMS (`n = 25 780`) лучший лаг **−2 ч** (`ρ = 0.556`), в сезонах `2022/23` (`n = 2 932`) и `2023/24` (`n = 4 268`) — **−1 ч** (`0.452` и `0.466`), в `2024/25` (`n = 4 382`) — **−3 ч** (`0.477`).
- **Сглаженные за 24 ч (без графика):** за весь период (`n = 25 561`) лучший лаг **−2 ч** (`0.772`), по сезонам — **−4 ч** в `2022/23` (`0.598`, `n = 2 822`), **+1 ч** в `2023/24` (`0.638`, `n = 4 251`) и **−3 ч** в `2024/25` (`0.696`, `n = 4 392`). Однако по всем лагам −12…+12 за весь период ρ лежит в узком диапазоне `0.728–0.772` (`0.768` на −6, `0.770` на 0, `0.754` на +6): кривая почти плоская и **версии меток не различает**.
- **Что это значит для версий меток (часовые ряды):**
  - Версия **«метки = местное время, записанное как UTC» (+6 ч)** отвергается: при сдвиге `+6` ч ρ резко падает во всех периодах — до `0.377` за весь период (против `0.556` в лучшем лаге), `0.284` в `2022/23`, `0.172` в `2023/24` и `0.190` в `2024/25`.
  - Версию **«−6 ч»** одними лагами со CAMS отличить от настоящего UTC **нельзя**: при `−6` ч ρ за весь период равна `0.531` (почти столько же, сколько `0.534` на лаге `0`), а в `2024/25` значения на `−6` и `0` вообще совпадают (`0.451` и `0.451`; в `2022/23` — `0.363` против `0.449`, в `2023/24` — `0.387` против `0.460`). Это происходит потому, что суточные пики станции (днём в 14–16 ч, шаг 13) и CAMS (вечером) разведены примерно на 6 часов. Различить версии `0` и `−6` проверим в шаге 17 по фазе профиля относительно температуры и CAMS.

![График 12](../figures/12_lag_check.png)

## Шаг 17. Насколько пик PM2.5 позже пика температуры и CAMS — по сезонам

Сравниваем суточные профили по часу UTC (`hour_utc`): у температуры Open-Meteo и модели CAMS время гарантированно в UTC, а сам `hour_utc` — это исходная метка часа без перевода в местный пояс. Для каждого отопительного сезона из `SEASONS` считаем круговой сдвиг профиля (`eda.profile_shift(..., max_shift=12)`) между `pm25` и `temp_c`, а для сезонов с полными 24 часами CAMS — также относительно `cams_pm25`.

Знак сдвига: `shift = +k` означает, что профиль `pm25` запаздывает на `k` часов относительно эталона (его пик позже на `k` ч); `shift = −k` — опережает на `k` часов. Дополнительно печатаем корреляции на сдвигах `−6`, `0` и `+6` ч, чтобы проверить версию `−6` ч, которую не смогли различить лаги в шаге 16. Также смотрим на сглаженный пик и минимум `pm25` по `hour_utc`: **если метки местные**, то настоящим местным часом был бы сам `hour_utc`.

In [21]:
phase_rows = []
cams_check_rows = []

for label in SEASONS:
    season_rows = heating[heating["season_label"] == label]
    pm25_utc = season_rows.groupby("hour_utc")["pm25"].mean()
    temp_utc = season_rows.groupby("hour_utc")["temp_c"].mean()
    cams_utc = season_rows.groupby("hour_utc")["cams_pm25"].mean()

    # Сглаженные экстремумы по hour_utc (= местный час при версии «метки местные»)
    pm25_smooth = eda.smooth_profile(pm25_utc)
    temp_smooth = eda.smooth_profile(temp_utc)

    temp_shifts = eda.profile_shift(temp_utc, pm25_utc, max_shift=12).set_index("shift")["corr"]
    best_temp_shift = int(temp_shifts.idxmax())

    has_cams = len(cams_utc) == 24 and cams_utc.notna().all()
    if has_cams:
        cams_smooth = eda.smooth_profile(cams_utc)
        cams_shifts = eda.profile_shift(cams_utc, pm25_utc, max_shift=12).set_index("shift")["corr"]
        best_cams_shift = int(cams_shifts.idxmax())
        cams_best_corr = float(cams_shifts.loc[best_cams_shift])
        cams_peak_utc = int(cams_smooth.idxmax())
        cams_check_rows.append(
            {
                "season": label,
                "cams_utc_peak": cams_peak_utc,
                "corr_minus_6": float(cams_shifts.loc[-6]),
                "corr_0": float(cams_shifts.loc[0]),
                "corr_plus_6": float(cams_shifts.loc[6]),
            }
        )
    else:
        best_cams_shift = pd.NA
        cams_best_corr = float("nan")
        cams_peak_utc = pd.NA

    phase_rows.append(
        {
            "season": label,
            "pm25_utc_peak": int(pm25_smooth.idxmax()),
            "pm25_utc_low": int(pm25_smooth.idxmin()),
            "temp_utc_peak": int(temp_smooth.idxmax()),
            "temp_shift": best_temp_shift,
            "temp_corr": float(temp_shifts.loc[best_temp_shift]),
            "temp_corr_minus_6": float(temp_shifts.loc[-6]),
            "temp_corr_0": float(temp_shifts.loc[0]),
            "temp_corr_plus_6": float(temp_shifts.loc[6]),
            "cams_utc_peak": cams_peak_utc,
            "cams_shift": best_cams_shift,
            "cams_corr": cams_best_corr,
        }
    )

phase_table = pd.DataFrame(phase_rows).set_index("season")
print("фаза суточного профиля по hour_utc (pm25 относительно temp_c и cams_pm25):")
print(phase_table.round(3).to_string())

cams_shift_checks = pd.DataFrame(cams_check_rows).set_index("season")
print("\nкорреляция профиля pm25 с cams_pm25 при сдвигах −6, 0, +6 ч:")
print(cams_shift_checks.round(3).to_string())

фаза суточного профиля по hour_utc (pm25 относительно temp_c и cams_pm25):
         pm25_utc_peak  pm25_utc_low  temp_utc_peak  temp_shift  temp_corr  temp_corr_minus_6  temp_corr_0  temp_corr_plus_6 cams_utc_peak cams_shift  cams_corr
season                                                                                                                                                          
2020/21             14             1              9           4      0.765             -0.705        0.565             0.669          <NA>       <NA>        NaN
2021/22             11             0              9           1      0.929             -0.370        0.889             0.313          <NA>       <NA>        NaN
2022/23              9             0              9           1      0.928             -0.421        0.898             0.303            14         -1      0.831
2023/24              9            23              9           1      0.961             -0.309        0.940             0

**Что видим.** (вывод шага 17) Сдвиг суточного профиля `pm25` относительно температуры меняется от сезона к сезону, повторяя картину дрейфа из шага 14:
- **Относительно температуры (`temp_c`, пик стабильно в 9 UTC во все пять сезонов):** в `2020/21` лучший сдвиг составляет **+4 ч** (`corr = 0.765`), в `2021/22`–`2023/24` он сокращается до **+1 ч** (`0.929`, `0.928`, `0.961`), а в `2024/25` — до **0 ч** (`0.981`). Сглаженный пик `pm25` по `hour_utc` за те же годы сместился с 14 UTC (`2020/21`) до 11 UTC (`2021/22`) и 9 UTC (`2022/23`–`2024/25`).
- **Проверка версий меток по температуре:**
  - В сезонах `2021/22`–`2024/25` при сдвиге `0` корреляция с температурой равна `0.889–0.981`, тогда как при сдвиге `+6` она падает до `0.313`, `0.303`, `0.169` и `0.053` — версия «метки местные (+6 ч)» отвергается. В раннем сезоне `2020/21` результат неоднозначный: при `+6` (`0.669`) корреляция сопоставима с `0` (`0.565`), а максимум — при `+4` (`0.765`).
  - При сдвиге `−6` ч корреляция с температурой отрицательна во всех пяти сезонах: `−0.705` (`2020/21`), `−0.370` (`2021/22`), `−0.421` (`2022/23`), `−0.309` (`2023/24`) и `−0.137` (`2024/25`). По фазе к температуре версия «−6 ч» не подтверждается.
- **Относительно `cams_pm25` (сезоны `2022/23`–`2024/25`, пик CAMS в 14–15 UTC):** лучший сдвиг меняется с **−1 ч** (`0.831` в `2022/23`) до **−5 ч** (`0.604` в `2023/24`) и **−6 ч** (`0.657` в `2024/25`). В последних двух сезонах корреляция с CAMS при `−6` ч (`0.586` и `0.657`) выше, чем при `0` (`0.347` и `0.253`), то есть по фазе к CAMS в `2023/24`–`2024/25` версия «−6 ч» подтверждалась бы (в `2022/23` — `0.342` против `0.825`), поскольку дневной пик станции (9 UTC) и вечерний пик модели CAMS (15 UTC) разнесены примерно на 6 часов. Таким образом, опора на температуру и опора на CAMS дают противоположный результат; выбор опоры является нашим допущением, поэтому решающего доказательства против версии «−6 ч» нет.
- **Справка: часы пика при версии «метки местные»** (если считать местным часом `hour_utc`): сглаженный пик PM2.5 приходился бы на **14:00** в `2020/21`, **11:00** в `2021/22` и **9:00** в `2022/23`–`2024/25`, а минимум — на **23:00–1:00** (`1` UTC в `2020/21`, `0` UTC в `2021/22`, `2022/23` и `2024/25`, `23` UTC в `2023/24`). Отвергается эта версия по корреляциям на сдвиге `+6` ч (в фазе к температуре в `2021/22`–`2024/25` и в лагах со CAMS из шага 16).

## Итог части 2

1. **Согласуются ли метки времени станции с настоящим UTC.** Проверки согласуются не полностью.
   - В таблице 34 106 часов со смещением +6 ч и 14 969 — с +5 ч; час перехода на UTC+5 — `2024-02-29 18:00:00+00:00` (вывод шага 12).
   - **Версия «метки = местное время (+6 ч)» отвергнута обеими опорами (лагами со CAMS и фазой к температуре).** В часовых лагах со CAMS за весь период (`n = 25 780`, вывод шага 16) корреляция на лаге `+6` падает до `0.377` (против `0.534` на `0` и `0.556` на лучшем лаге `−2`), а по сезонам — до `0.172–0.284`. По фазе профиля с температурой (вывод шага 17) в сезонах `2021/22`–`2024/25` при сдвиге `0` корреляция равна `0.889–0.981` против `0.053–0.313` при `+6` (неоднозначен только ранний сезон `2020/21`: `0.565` на `0`, `0.669` на `+6`, `0.765` на `+4`). Справочно: при версии «метки местные» пик в `2022/23`–`2024/25` приходился бы на `9:00` (`14:00` в `2020/21`, `11:00` в `2021/22`), а минимум — на `23:00–1:00` (вывод шага 17).
   - **Версия «−6 ч»:** на часовых лагах со CAMS за весь период `ρ` при `−6` равна `0.531` и почти не отличается от `0.534` на `0`, а в сезоне `2024/25` они совпадают (`0.451` и `0.451`, вывод шага 16); сглаженный 24-часовой вариант неинформативен (`ρ = 0.728–0.772` по всем лагам, вывод шага 16). По фазе к температуре версия «−6 ч» не подтверждается: корреляция при `−6` ч отрицательна во всех пяти сезонах (`−0.705` … `−0.137`, вывод шага 17). Однако по фазе к CAMS в `2023/24`–`2024/25` версия «−6 ч» подтверждалась бы (`0.586` и `0.657` при `−6` против `0.347` и `0.253` при `0`; в `2022/23` — `0.342` против `0.825`, вывод шага 17). Выбор опоры является нашим допущением, поэтому решающего доказательства против «−6 ч» нет.
   - **Переход 1 марта 2024 (узкое 29-суточное окно, минимум 28 наблюдений `pm25` на час):** контрольные ряды `temp_c` (`−1` ч, `0.973`) и `cams_pm25` (`−1` ч, `0.988`) сдвинулись по предсказанию, а станция `pm25` показала лучший сдвиг **−2 ч** (`0.942` против `0.871` на `−1` и `0.724` на `0`, вывод шага 15). Сдвиг **−2 ч** у станции не совпадает ни с одним предсказанием для верного UTC (`0` или `−1` ч) и остаётся необъяснённым расхождением. Сравнение за весь отопительный период до и после перехода (`−2` ч у `pm25`, `0.988`) непригодно, так как смешивает перевод часов с дрейфом профиля между годами (вывод шага 15).
   - **Ограничение:** версия **+6 ч** отвергнута; версия **−6 ч** не решена (опоры температура и CAMS дают противоположный результат); сдвиг на **1–3 ч** этими проверками не различим (включая ±1 ч для обозначения начала или конца часа, отмеченное в документации OpenAQ). Данные в `data/` не менялись.

2. **Насколько устойчив суточный профиль по отопительным сезонам.**
   - Порог `>= 60` значений в часе прошли пять сезонов (`2020/21`–`2024/25`, минимум `121–182` значения в часе; сезоны `2019/20` с минимумом `6` и `2025/26` с `30` отброшены, вывод шага 12).
   - Профиль **не устойчив** по часу пика и форме (вывод шага 13): сглаженный пик сместился с **20:00** (`80.7` µg/m³, сырой `idxmax` `19`) в `2020/21` к **17:00** (`61.6`, сырой `17`) в `2021/22`, **15:00** (`71.5`, сырой `16`) в `2022/23`, **16:00** (`52.4`, сырой `16`) в `2023/24` и **14:00** (`60.2`, сырой `13`) в `2024/25`. Минимум держится утром в **5:00–7:00**, а отношение `peak_to_low` выросло с `1.7` в `2020/21` до `2.4–2.7` в последующих сезонах (вывод шага 13).
   - Корреляция Пирсона профилей между сезонами `2021/22`–`2023/24` высокая (`0.98–0.99`, максимум `0.988` между `2021/22` и `2023/24`), но между `2020/21` и `2024/25` падает до `0.515` (вывод шага 13).
   - По кварталам в часах UTC (вывод шага 14): осенне-зимний пик сдвинулся с **14–15 UTC** (`2020Q4`–`2021Q1`) через **12 UTC** (`2021Q4`) к **8–11 UTC** (`2022Q1`–`2025Q4`), минимум оставался в **23–1 UTC**. Смена пика уже шла в `2021Q4`–`2022Q1` — **до** отброшенного пропуска `2022Q2`–`2022Q3` (`n = 668` и `552`).

3. **Что можно утверждать про гипотезу H3 в разделе 5.1.**
   - Версия **+6 ч** отвергнута; версия **−6 ч** не решена (опоры температура и CAMS дают противоположный результат); сдвиг на **1–3 ч** этими проверками не различим. Из-за смещения профиля по годам нельзя формулировать H3 через общий средний профиль с фиксированным часом пика «14:00» (выводы шагов 13–14).
   - В разделе **5.1** гипотезу **H3** проверяем **отдельно по каждому из пяти полных отопительных сезонов (`2020/21`–`2024/25`)**: для сезона `2020/21` час суточного цикла ненадёжен (фаза к температуре `+4` ч, `0.765`, вывод шага 17), поэтому по нему утверждаем только наличие цикла и размах `peak_to_low` (`1.7`, вывод шага 13), а положение пика и минимума по часам — только по сезонам `2021/22`–`2024/25` (`peak_to_low = 2.4–2.7`, вывод шага 13), без привязки к единому часу пика; часы пика и минимума в H3 указываем при допущении, что метки — настоящий UTC (версия −6 ч не исключена).
   - Причину смены формы профиля (изменения в городе или изменение записи меток) по этим данным не знаем: зимний пик по UTC сместился с `14–15 UTC` (`2020Q4`–`2021Q1`) на `8–9 UTC` с `2022Q1` (`8`–`9` UTC в `2022Q1`, `2022Q4`, `2024Q1`, `2024Q4`, `2025Q1`, `2025Q4`; `11` в `2023Q1` и `10` в `2023Q4`), то есть сдвиг составил ≈ 5–7 ч (по кварталам: `Q4` `14→9/9/8`, `Q1` `15→9/8/8`, вывод шага 14).

Раздел тестов (5.1) будет добавлен ниже.